# Daily Challenge: Interactive Data Visualization with Matplotlib and Seaborn

---

### 👩‍🏫 What You'll Learn

- Advanced data visualization techniques.
- Interactive chart creation using Matplotlib.
- Elegant static data presentation with Seaborn.

---

### Tasks Overview
1. **Data Preparation** — Download, explore, clean, and preprocess the US Superstore dataset.
2. **Matplotlib Visualizations** — Sales trends over years (line chart) & sales by region (map).
3. **Seaborn Visualizations** — Top 10 products by sales (bar chart) & profit vs discount (scatter plot).
4. **Comparative Analysis** — Compare insights, ease of use, and effectiveness of both libraries.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests
import io
import warnings

warnings.filterwarnings("ignore")

# Set default styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Libraries imported successfully!")

## 2. Load and Explore the Dataset

We download the **US Superstore** dataset directly from GitHub and load it into a pandas DataFrame.

In [ ]:
# Download the US Superstore dataset from GitHub
url = "https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%205%20-%20Data%20Processing/W5D5%20-%20Mini-project%20-%20bis/US%20Superstore%20data.xls"

response = requests.get(url)
df = pd.read_excel(io.BytesIO(response.content))

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
df.head()

In [ ]:
# Basic exploration
print("=== Basic Statistics ===")
print(df.describe())
print(f"\n=== Missing Values ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Data Cleaning and Preprocessing

- Parse date columns properly.
- Drop rows with critical missing values.
- Extract the **Year** from `Order Date` for temporal analysis.

In [ ]:
# Convert date columns to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# Extract year and month from Order Date
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month

# Drop rows missing key numeric fields
df.dropna(subset=["Sales", "Profit", "Discount"], inplace=True)

# Strip whitespace from string columns
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

print(f"Cleaned dataset shape: {df.shape}")
print(f"\nYear range: {df['Year'].min()} – {df['Year'].max()}")
print(f"Unique categories: {df['Category'].unique()}")
df.head(3)

## 4. Sales Trends Over the Years — Matplotlib Line Chart

We aggregate total sales per year and visualize the trend with an annotated interactive-style line chart using Matplotlib.

In [ ]:
# Aggregate total sales by year
sales_by_year = df.groupby("Year")["Sales"].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    sales_by_year["Year"],
    sales_by_year["Sales"],
    marker="o",
    linewidth=2.5,
    color="#2196F3",
    markersize=8,
    label="Total Sales",
)

# Annotate each data point with its value
for _, row in sales_by_year.iterrows():
    ax.annotate(
        f"${row['Sales']:,.0f}",
        xy=(row["Year"], row["Sales"]),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color="#333333",
    )

ax.set_title("Annual Sales Trend — US Superstore", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Total Sales (USD)", fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_xticks(sales_by_year["Year"])
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

print("\nInsight: Sales have grown consistently year-over-year, indicating strong business momentum.")

## 5. Sales Distribution by State — Matplotlib Bar Map (Horizontal Bar Chart)

Since Matplotlib does not natively support geographic maps, we use a **ranked horizontal bar chart** as a proxy to visualize sales distribution across US states, ordered from highest to lowest.

In [ ]:
# Aggregate sales by state and sort
sales_by_state = (
    df.groupby("State")["Sales"]
    .sum()
    .sort_values(ascending=True)
)

# Use a color gradient based on sales volume
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(sales_by_state)))

fig, ax = plt.subplots(figsize=(12, 14))
bars = ax.barh(sales_by_state.index, sales_by_state.values, color=colors)

# Add value labels to each bar
for bar, val in zip(bars, sales_by_state.values):
    ax.text(
        bar.get_width() + 1000,
        bar.get_y() + bar.get_height() / 2,
        f"${val:,.0f}",
        va="center",
        fontsize=7,
        color="#333333",
    )

ax.set_title("Sales Distribution by State — US Superstore", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Total Sales (USD)", fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInsight: California, New York, and Texas dominate sales, reflecting larger population centers.")

## 6. Top 10 Products by Sales — Seaborn Bar Chart

Using Seaborn, we identify and plot the **10 best-selling products** by total sales.

In [ ]:
# Compute total sales per product and select top 10
top10_products = (
    df.groupby("Product Name")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))

sns.barplot(
    data=top10_products,
    x="Sales",
    y="Product Name",
    palette="Blues_r",
    ax=ax,
)

# Add value labels inside bars
for container in ax.containers:
    ax.bar_label(container, fmt="$%.0f", padding=5, fontsize=9)

ax.set_title("Top 10 Products by Total Sales — US Superstore", fontsize=15, fontweight="bold", pad=12)
ax.set_xlabel("Total Sales (USD)", fontsize=12)
ax.set_ylabel("Product Name", fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()

print("\nInsight: A small set of high-value products (e.g., Canon copiers, Cisco phones) drives a disproportionate share of total revenue — classic Pareto distribution.")

## 7. Profit vs Discount Analysis — Seaborn Scatter Plot

We explore whether higher discounts negatively impact profit, broken down by **product category**.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.scatterplot(
    data=df,
    x="Discount",
    y="Profit",
    hue="Category",
    alpha=0.55,
    s=40,
    palette="Set2",
    ax=ax,
)

# Add a reference line at Profit = 0
ax.axhline(0, color="red", linestyle="--", linewidth=1.2, label="Break-even (Profit = 0)")

ax.set_title("Profit vs Discount by Category — US Superstore", fontsize=15, fontweight="bold", pad=12)
ax.set_xlabel("Discount Rate", fontsize=12)
ax.set_ylabel("Profit (USD)", fontsize=12)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(title="Category", loc="upper right")
plt.tight_layout()
plt.show()

print("\nInsight: There is a clear negative correlation between discount rate and profit.")
print("Orders with discounts above ~40% frequently result in losses (negative profit).")
print("The Furniture category is especially sensitive to high discounts.")

## 8. Comparative Analysis: Matplotlib vs Seaborn

---

### Matplotlib

| Aspect | Assessment |
|---|---|
| **Flexibility** | Very high — full control over every visual element (axes, ticks, annotations, colors, layout). |
| **Verbosity** | High — requires more lines of code to produce polished visuals. |
| **Best for** | Custom, highly tailored plots; publication-quality figures; embedding in dashboards. |
| **Learning curve** | Steeper — you must understand the figure/axes object model. |
| **Interactive feel** | Achievable with `mplcursors` or `ipywidgets`, but not built-in. |

---

### Seaborn

| Aspect | Assessment |
|---|---|
| **Flexibility** | Moderate — sensible defaults; deeper customization still requires Matplotlib. |
| **Verbosity** | Low — beautiful charts in just a few lines. |
| **Best for** | Exploratory data analysis; statistical plots (distributions, correlations, categorical). |
| **Learning curve** | Gentle — high-level API built on top of Matplotlib. |
| **Integration** | Returns Matplotlib `Axes`, so you can always refine with Matplotlib afterwards. |

---

### Key Takeaways

- **Use Matplotlib** when you need precise control, custom layouts, or need to embed charts in a larger figure.
- **Use Seaborn** for fast, aesthetically pleasing EDA charts with minimal code.
- In practice, **combining both** is the most effective approach: use Seaborn for the heavy lifting and Matplotlib for final polish.
- Both libraries revealed consistent insights: sales are growing year-over-year, a few top products dominate revenue, and aggressive discounting hurts profitability — especially in the Furniture category.